Our `State` will have
1. Runs
2. Balls
3. Fours
4. Sixes
5. Strike_rate
6. Boundry_percentage (Percentage of runs in boundry.)
7. Balls_per_boundry (If played 10 balls and 4 boundries, means on every 2.5th ball, battter hits a boundry)
8. Summary

In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated, Optional, TypedDict

import operator

In [60]:
class BatterState(BaseModel):

    runs: Annotated[int, Field(description = "Runs made by the batter")]
    balls: Annotated[int, Field(description = "Balls played by the batter")]
    fours: Annotated[int, Field(description = "No_of_Fours by the batter")]
    sixes: Annotated[int, Field(description = "No_of_sixes by the batter")]

    strike_rate: Annotated[Optional[float], Field(description = "Strike rate of the batter", default = None), operator.add]
    boundry_percent: Annotated[Optional[float], Field(description = "Boundry Percentage", default = None), operator.add]
    balls_per_boundry: Annotated[Optional[float], Field(description = "Balls per boundry", default = None), operator.add]

    summary: Annotated[Optional[str], Field(description = "Summary of the batter", default = None), operator.add]


'''
class BatterState(TypedDict):

    runs: int
    balls: int 
    fours: int
    sixes: int
    
    strike_rate: float
    boundry_percent: float
    balls_per_boundry: float

    summary: str

'''

'\nclass BatterState(TypedDict):\n\n    runs: int\n    balls: int \n    fours: int\n    sixes: int\n\n    strike_rate: float\n    boundry_percent: float\n    balls_per_boundry: float\n\n    summary: str\n\n'

In [61]:
# Creating nodes (Functions)

def strike_rate(state: BatterState) -> BatterState:

    # Fetching required states
    runs = state.runs
    balls = state.balls

    # Calculating strike_rate
    strike_rate = (100*runs)/balls

    # Returning Strike_rate

    return {"strike_rate": round(strike_rate, 2)}

#----------------------------------------------------------------------------------------------------

def boundry_percent(state: BatterState) -> BatterState:

    # Fetching required states
    total_runs = state.runs
    runs_in_fours = state.fours * 4
    runs_in_sixes = state.sixes * 6

    total_boundry_runs = runs_in_fours + runs_in_sixes

    # Calculating boundry_percent
    boundry_percent = ((total_boundry_runs)/total_runs) * 100

    # Returning boundry_point
    return {"boundry_percent": round(boundry_percent)}

#----------------------------------------------------------------------------------------------------

def balls_per_boundry(state: BatterState) -> BatterState:

    # Fetching required states
    balls_played = state.balls
    no_of_fours = state.fours
    no_of_sixes = state.sixes

    total_boundries = no_of_fours + no_of_sixes

    # Calculating boundry_percent
    balls_per_boundry = balls_played//total_boundries

    # Returning boundry_point
    return {"balls_per_boundry": balls_per_boundry}

#----------------------------------------------------------------------------------------------------

def summary(state: BatterState) -> BatterState:

    summary = f"""Strike Rate: {state.strike_rate}
    Boundry Percentage: {state.boundry_percent}
    Balls Per Boundry: {state.balls_per_boundry}
    """

    return {"summary": summary}

In [62]:
# Creating the graph
graph = StateGraph(BatterState)

# Adding nodes
graph.add_node('strike_rate', strike_rate)
graph.add_node('boundry_percent', boundry_percent)
graph.add_node('balls_per_boundry', balls_per_boundry)
graph.add_node('summary', summary)

# Adding Edges
graph.add_edge(START, 'strike_rate')
graph.add_edge(START, 'boundry_percent')
graph.add_edge(START, 'balls_per_boundry')

graph.add_edge("strike_rate", 'summary')
graph.add_edge("boundry_percent", 'summary')
graph.add_edge("balls_per_boundry", 'summary')

graph.add_edge('summary', END)

# Compiling the graph
workflow = graph.compile()

In [34]:
workflow

KeyboardInterrupt: 

In [63]:
initial_state = {
    "runs": 100,
    "balls": 75,
    "fours": 5,
    "sixes": 6
}

response = workflow.invoke(initial_state)

In [64]:
response

{'runs': 100,
 'balls': 75,
 'fours': 5,
 'sixes': 6,
 'strike_rate': 133.33,
 'boundry_percent': 56,
 'balls_per_boundry': 6,
 'summary': 'Strike Rate: 133.33\n    Boundry Percentage: 56.0\n    Balls Per Boundry: 6.0\n    '}

In [65]:
round(((56/100)*100))

56

In [68]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png(max_retries = 5, retry_delay = 2.0))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

# In next noteboos, we will use LLM in Parallel workflow.
# Go to Inrtoduction file first and then move to next notebook.

# ***Thankyou***